In [0]:
df_smp = spark.table("workspace.default.silver_simples_nacional")
df_emp = spark.table("workspace.default.silver_empresas")

df_gold_simples = (
    df_smp.alias("smp")
    .join(df_emp.alias("emp"), "nr_cnpj_basico", "inner")
    .select(
        F.col("smp.nr_cnpj_basico"),
        F.col("emp.nm_razao_social"),
        F.when(F.col("smp.fl_opcao_simples") == "S", True).otherwise(False).alias("fl_optante_simples"),
        F.col("smp.dt_opcao_simples"),
        F.col("smp.dt_exclusao_simples"),
        F.when(F.col("smp.fl_opcao_mei") == "S", True).otherwise(False).alias("fl_optante_mei"),
        F.col("smp.dt_opcao_mei"),
        F.col("smp.dt_exclusao_mei"),
        # Métricas derivadas de duração em dias
        F.when(
            F.col("smp.dt_exclusao_simples").isNotNull(),
            F.datediff(F.col("smp.dt_exclusao_simples"), F.col("smp.dt_opcao_simples"))
        ).when(
            F.col("smp.fl_opcao_simples") == "S",
            F.datediff(F.current_date(), F.col("smp.dt_opcao_simples"))
        ).otherwise(0).alias("nu_dias_permanencia_simples"),
        F.col("emp.vl_capital_social"),
        F.current_timestamp().alias("dt_carga_gold")
    )
)

(
    df_gold_simples.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_dp_regimes_tributarios_simples_mei")
)
print("-> workspace.default.gold_dp_regimes_tributarios_simples_mei gravada.")